# Scraping referendum per sezione

Questo notebook e' uno scheletro operativo per scaricare i CSV dalle pagine comunali di Eligendo.

Workflow previsto:
1. Leggere i link da `linkdownload_sezioni.txt`.
2. Aprire ogni pagina con un browser automatizzato.
3. Cercare il bottone `Esporta CSV`.
4. Salvare il file scaricato in una cartella locale.

Nota: non lo sto eseguendo qui. Va rifinito e lanciato in locale da te.

## Dipendenze suggerite

Qui la base e' Selenium con Firefox:

```bash
pip install pandas selenium jupyter
```

Serve anche Firefox installato nel sistema. Se `geckodriver` non e' nel PATH, dovrai indicarlo a mano nella funzione `build_firefox_driver`.

In [47]:
from __future__ import annotations

import json
import re
import time
from pathlib import Path
from typing import Dict

import pandas as pd

from selenium import webdriver
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.options import Options as FirefoxOptions
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

ROOT = Path.cwd()
LINKS_FILE = ROOT / "linkdownload_sezioni.txt"
DOWNLOAD_DIR = ROOT / "downloads_csv"
DOWNLOAD_DIR.mkdir(exist_ok=True)

DATE_TAG = "2026-03-22"

In [48]:
def parse_links_file(path: Path) -> Dict[str, str]:
    """Parsa il file testuale tipo:
    "roma":"https://...",
    "milano":"https://..."
    """
    text = path.read_text(encoding="utf-8")
    pairs = re.findall(r'"([^"]+)"\s*:\s*"([^"]+)"', text)
    if not pairs:
        raise ValueError(f"Nessun link trovato in {path}")
    return dict(pairs)


city_links = parse_links_file(LINKS_FILE)
pd.DataFrame(city_links.items(), columns=["city", "url"])

,city,url
0,roma,https://elezioni.interno.gov.it/risultati/2026...
1,milano,https://elezioni.interno.gov.it/risultati/2026...
2,napoli,https://elezioni.interno.gov.it/risultati/2026...
3,bologna,https://elezioni.interno.gov.it/risultati/2026...
4,torino,https://elezioni.interno.gov.it/risultati/2026...
5,genova,https://elezioni.interno.gov.it/risultati/2026...
6,firenze,https://elezioni.interno.gov.it/risultati/2026...
7,palermo,https://elezioni.interno.gov.it/risultati/2026...
8,reggiocalabria,https://elezioni.interno.gov.it/risultati/2026...
9,taranto,https://elezioni.interno.gov.it/risultati/2026...


## Test rapido su una singola citta'

Questa funzione prova a:
- aprire la pagina;
- aspettare il rendering;
- trovare un elemento con testo `Esporta CSV`;
- scaricare il file.

Se il sito cambia struttura, i selettori da ritoccare saranno soprattutto dentro `candidate_xpaths`.

In [49]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

In [53]:
def slugify(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", value.lower()).strip("_")


def build_firefox_driver(download_dir: Path, headless: bool = True) -> webdriver.Firefox:
    options = FirefoxOptions()
    if headless:
        options.add_argument("-headless")

    options.set_preference("browser.download.folderList", 2)
    options.set_preference("browser.download.dir", str(download_dir.resolve()))
    options.set_preference("browser.download.useDownloadDir", True)
    options.set_preference("browser.helperApps.neverAsk.saveToDisk", "text/csv,application/csv,application/octet-stream")
    options.set_preference("browser.download.manager.showWhenStarting", False)
    options.set_preference("pdfjs.disabled", True)
    options.set_preference("browser.download.alwaysOpenPanel", False)

    # Se serve un geckodriver esplicito, puoi usare:
    # from selenium.webdriver.firefox.service import Service
    # return webdriver.Firefox(service=Service(r'C:\\percorso\\geckodriver.exe'), options=options)
    driver = webdriver.Firefox(options=options)
    driver.maximize_window()
    return driver


def wait_for_new_csv(download_dir: Path, existing_files: set[Path], timeout_s: int = 60) -> Path:
    wait = WebDriverWait(object(), timeout_s, poll_frequency=1)

    def _download_completed(_driver_placeholder):
        current_files = set(download_dir.glob("*.csv"))
        new_files = sorted(current_files - existing_files, key=lambda p: p.stat().st_mtime)
        partial_files = list(download_dir.glob("*.part"))
        if new_files and not partial_files:
            return new_files[-1]
        return False

    return wait.until(_download_completed)
def download_city_csv(city: str, url: str, headless: bool = True, timeout_s: int = 45) -> Path:
    destination = DOWNLOAD_DIR / f"{DATE_TAG}_{slugify(city)}.csv"
    temp_destination = DOWNLOAD_DIR / f"{DATE_TAG}_{slugify(city)}__new.csv"
    driver = build_firefox_driver(DOWNLOAD_DIR, headless=headless)

    try:
        wait = WebDriverWait(driver, timeout_s)
        existing_files = set(DOWNLOAD_DIR.glob("*.csv"))

        driver.get(url)
        wait.until(lambda d: d.execute_script("return document.readyState") == "complete")
        time.sleep(7)

        candidate_xpaths = [
            "//*[contains(concat(' ', normalize-space(@class), ' '), ' q-btn__content ') and contains(concat(' ', normalize-space(@class), ' '), ' text-center ') and contains(concat(' ', normalize-space(@class), ' '), ' q-anchor--skip ') and contains(translate(normalize-space(.), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'esporta in csv')]",
            "//button//*[contains(concat(' ', normalize-space(@class), ' '), ' q-btn__content ') and contains(translate(normalize-space(.), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'esporta in csv')]",
            "//a//*[contains(concat(' ', normalize-space(@class), ' '), ' q-btn__content ') and contains(translate(normalize-space(.), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'esporta in csv')]",
            "//*[contains(translate(normalize-space(.), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'esporta in csv')]",
        ]

        trigger = None
        time.sleep(4)
        for xpath in candidate_xpaths:
            matches = driver.find_elements(By.XPATH, xpath)
            visible_matches = [element for element in matches if element.is_displayed()]
            if visible_matches:
                trigger = visible_matches[0]
                break

        if trigger is None:
            driver.save_screenshot(str(DOWNLOAD_DIR / f"debug_{slugify(city)}.png"))
            raise RuntimeError(f"Bottone 'Esporta CSV' non trovato per {city}")

        time.sleep(2)
        wait.until(EC.element_to_be_clickable(trigger))
        time.sleep(1)
        trigger.click()

        downloaded_file = wait_for_new_csv(DOWNLOAD_DIR, existing_files, timeout_s=60)
        if temp_destination.exists():
            temp_destination.unlink()
        downloaded_file.replace(temp_destination)
        temp_destination.replace(destination)
        return destination

    except TimeoutException as exc:
        driver.save_screenshot(str(DOWNLOAD_DIR / f"timeout_{slugify(city)}.png"))
        raise RuntimeError(f"Timeout su {city}: {url}") from exc

    finally:
        driver.quit()


In [54]:
# Esempio: prova prima su una sola citta'.
# headless=False ti aiuta a vedere i passaggi nel browser Firefox.
path_roma = download_city_csv("roma", city_links["roma"], headless=False)
path_roma

WindowsPath('c:/Users/gabri/Documents/GitHub/referendum2026_mappepersezione/downloads_csv/2026-03-22_roma.csv')

## Batch su tutte le citta'

Questa cella salva i risultati e continua anche se una citta' fallisce.

In [55]:
results = []

for city, url in city_links.items():
#for city, url in list(city_links.items())[1:]

    try:
        saved_path = download_city_csv(city, url, headless=False)
        results.append({"city": city, "url": url, "status": "ok", "file": str(saved_path)})
    except Exception as exc:
        results.append({"city": city, "url": url, "status": "error", "file": None, "error": str(exc)})

results_df = pd.DataFrame(results)
results_df

,city,url,status,file
0,roma,https://elezioni.interno.gov.it/risultati/2026...,ok,c:\Users\gabri\Documents\GitHub\referendum2026...
1,milano,https://elezioni.interno.gov.it/risultati/2026...,ok,c:\Users\gabri\Documents\GitHub\referendum2026...
2,napoli,https://elezioni.interno.gov.it/risultati/2026...,ok,c:\Users\gabri\Documents\GitHub\referendum2026...
3,bologna,https://elezioni.interno.gov.it/risultati/2026...,ok,c:\Users\gabri\Documents\GitHub\referendum2026...
4,torino,https://elezioni.interno.gov.it/risultati/2026...,ok,c:\Users\gabri\Documents\GitHub\referendum2026...
5,genova,https://elezioni.interno.gov.it/risultati/2026...,ok,c:\Users\gabri\Documents\GitHub\referendum2026...
6,firenze,https://elezioni.interno.gov.it/risultati/2026...,ok,c:\Users\gabri\Documents\GitHub\referendum2026...
7,palermo,https://elezioni.interno.gov.it/risultati/2026...,ok,c:\Users\gabri\Documents\GitHub\referendum2026...
8,reggiocalabria,https://elezioni.interno.gov.it/risultati/2026...,ok,c:\Users\gabri\Documents\GitHub\referendum2026...
9,taranto,https://elezioni.interno.gov.it/risultati/2026...,ok,c:\Users\gabri\Documents\GitHub\referendum2026...


## Controllo veloce dei CSV scaricati

Quando avrai i file, questa cella aiuta a ispezionare intestazioni e prime righe.

In [56]:
csv_files = sorted(DOWNLOAD_DIR.glob("*.csv"))
csv_files[:10]

[WindowsPath('c:/Users/gabri/Documents/GitHub/referendum2026_mappepersezione/downloads_csv/2026-03-22_bologna.csv'),
 WindowsPath('c:/Users/gabri/Documents/GitHub/referendum2026_mappepersezione/downloads_csv/2026-03-22_firenze.csv'),
 WindowsPath('c:/Users/gabri/Documents/GitHub/referendum2026_mappepersezione/downloads_csv/2026-03-22_genova.csv'),
 WindowsPath('c:/Users/gabri/Documents/GitHub/referendum2026_mappepersezione/downloads_csv/2026-03-22_milano.csv'),
 WindowsPath('c:/Users/gabri/Documents/GitHub/referendum2026_mappepersezione/downloads_csv/2026-03-22_napoli.csv'),
 WindowsPath('c:/Users/gabri/Documents/GitHub/referendum2026_mappepersezione/downloads_csv/2026-03-22_palermo.csv'),
 WindowsPath('c:/Users/gabri/Documents/GitHub/referendum2026_mappepersezione/downloads_csv/2026-03-22_reggiocalabria.csv'),
 WindowsPath('c:/Users/gabri/Documents/GitHub/referendum2026_mappepersezione/downloads_csv/2026-03-22_roma.csv'),
 WindowsPath('c:/Users/gabri/Documents/GitHub/referendum2026_ma

In [ ]:
# if csv_files:
#     sample_df = pd.read_csv(csv_files[0])
#     display(sample_df.head())
#     print(sample_df.columns.tolist())
# else:
#     print("Nessun CSV disponibile ancora.")

ParserError: Error tokenizing data. C error: Expected 1 fields in line 3, saw 3


## Esporta JSON per il sito

Questa cella converte i CSV di affluenza in file JSON leggeri, piu' semplici da usare nel frontend.

In [58]:
DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

def parse_turnout_csv(csv_path: Path) -> list[dict]:
    rows = []
    lines = [line.strip() for line in csv_path.read_text(encoding="utf-8-sig").splitlines() if line.strip()]
    if len(lines) < 3:
        return rows

    headers = [item.strip().strip('"') for item in lines[1].split(';')]
    ente_idx = headers.index("Ente")
    turnout_columns = {}
    for idx, header in enumerate(headers):
        match = re.match(r"^% ore (\d+)$", header, flags=re.I)
        if match:
            turnout_columns[match.group(1)] = idx

    for line in lines[2:]:
        parts = [item.strip().strip('"') for item in line.split(';')]
        if len(parts) <= ente_idx:
            continue

        name = parts[ente_idx]
        match = re.match(r"^SEZIONE\s+(\d+)$", name, flags=re.I)
        if not match:
            continue

        turnout = {}
        for slot, idx in turnout_columns.items():
            turnout[slot] = float(parts[idx].replace(',', '.')) if idx < len(parts) and parts[idx] else None

        rows.append({
            "section": int(match.group(1)),
            "name": name,
            "turnout": turnout,
            "results": None,
        })

    return rows


exported = []
for city in city_links:
    csv_path = DOWNLOAD_DIR / f"{DATE_TAG}_{slugify(city)}.csv"
    if not csv_path.exists():
        continue

    sections = parse_turnout_csv(csv_path)
    turnout_slots = sorted({slot for row in sections for slot in row.get("turnout", {}).keys()}, key=int)
    payload = {
        "city": city,
        "updated_at": "2026-03-22T19:00:00+01:00",
        "turnout_slots": turnout_slots,
        "sections": sections,
    }
    json_path = DATA_DIR / f"turnout_{slugify(city)}.json"
    json_path.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")
    exported.append({"city": city, "json": str(json_path), "rows": len(payload["sections"])})

pd.DataFrame(exported)

,city,json,rows
0,roma,c:\Users\gabri\Documents\GitHub\referendum2026...,2569
1,milano,c:\Users\gabri\Documents\GitHub\referendum2026...,1225
2,napoli,c:\Users\gabri\Documents\GitHub\referendum2026...,869
3,bologna,c:\Users\gabri\Documents\GitHub\referendum2026...,437
4,torino,c:\Users\gabri\Documents\GitHub\referendum2026...,911
5,genova,c:\Users\gabri\Documents\GitHub\referendum2026...,646
6,firenze,c:\Users\gabri\Documents\GitHub\referendum2026...,354
7,palermo,c:\Users\gabri\Documents\GitHub\referendum2026...,593
8,reggiocalabria,c:\Users\gabri\Documents\GitHub\referendum2026...,192
9,taranto,c:\Users\gabri\Documents\GitHub\referendum2026...,189


In [59]:
import subprocess
from pathlib import Path

def build_city_bundles(root_path):
    root = Path(root_path)
    script = root / "build_city_bundles.ps1"

    result = subprocess.run(
        [
            "powershell",
            "-ExecutionPolicy",
            "Bypass",
            "-File",
            str(script),
        ],
        cwd=root,
        text=True,
        capture_output=True,
    )

    print(result.stdout)
    if result.stderr:
        print(result.stderr)

    result.check_returncode()

build_city_bundles(r"c:\Users\gabri\Documents\GitHub\referendum2026_mappepersezione")



Name                       Length
----                       ------
bologna.bundle.js         7205482
firenze.bundle.js         8042547
genova.bundle.js          7622244
milano.bundle.js         10519794
napoli.bundle.js          9993952
palermo.bundle.js        12816080
reggiocalabria.bundle.js 10129113
roma.bundle.js           36713272
taranto.bundle.js         6081883
torino.bundle.js          5439100





## Possibili aggiustamenti utili

- Se il click apre una nuova tab invece di un download diretto, dopo il `click()` controlla `driver.window_handles` e spostati sulla tab nuova.
- Se il bottone compare solo dopo aver scelto un livello territoriale, aggiungi i click intermedi prima della ricerca XPath.
- Se i file hanno nomi ambigui, rinominali con `citta + data` come gia' previsto sopra.
- Se alcune pagine falliscono, gli screenshot di debug vengono salvati in `downloads_csv/`.